<a href="https://colab.research.google.com/github/hyeonji-jang65/econ81230-project-jang/blob/main/ML_project_samplecheck.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pyreadstat

Mounted at /content/drive


In [ ]:
import numpy as np, pandas as pd, pyreadstat

PATH = '/content/drive/MyDrive/HRS/randhrs1992_2022v1.dta'

_, meta = pyreadstat.read_dta(PATH, metadataonly=True)
lower = {c.lower(): c for c in meta.column_names}

W = range(1, 17)
fixed = ['hhidpn', 'hacohort', 'rabyear', 'ragender', 'raedyrs', 'ravetrn']
stems = ['inw', 'peninc', 'peni_n', 'work', 'slfemp', 'samemp', 'jcpen', 'penct',
         'ptyp1', 'ptyp2', 'ptyp3', 'ptyp4',
         'dcbal1', 'dcbal2', 'dcbal3', 'dcbal4',
         'iearn', 'ipen', 'iirawy1', 'isret', 'jcten', 'jcocc',
         'agey_e', 'mstat', 'cendiv', 'wtresp']

def vname(stem, w):
    return f'inw{w}' if stem == 'inw' else f'r{w}{stem}'

cols = [lower[v] for v in fixed if v in lower]
cols += [lower[vname(s, w)] for s in stems for w in W if vname(s, w) in lower]

df, meta2 = pyreadstat.read_dta(PATH, usecols=cols)
df.columns = df.columns.str.lower()
labels = {k.lower(): v for k, v in meta2.variable_value_labels.items()}
print(df.shape)

(45234, 408)


In [ ]:
N = len(df)
idx = np.arange(N)

def A(stem):
    """(N, 17) array; column w = wave w (column 0 unused). Missing waves = NaN."""
    out = np.full((N, 17), np.nan)
    for w in W:
        n = vname(stem, w)
        if n in df:
            out[:, w] = df[n].to_numpy(dtype=float)
    return out

def at(arr, t):
    """Value of arr at wave t for each person (NaN if t < 1)."""
    out = np.full(N, np.nan)
    ok = (t >= 1) & (t <= 16)
    out[ok] = arr[idx[ok], t[ok]]
    return out

In [ ]:
pen, work = A('peninc'), A('work')

onset = np.full(N, -1)
for w in range(16, 1, -1):
    onset[pen[:, w] == 1] = w

prior0 = np.zeros(N, bool)
for w in range(2, 16):
    prior0 |= (pen[:, w] == 0) & (w < onset)

nxt = np.where((onset >= 1) & (onset <= 15), onset + 1, -1)
two = at(pen, nxt) == 1
t0 = np.where(two, onset, -1)

t1 = np.full(N, -1)
for w in range(1, 16):
    t1[(work[:, w] == 1) & (w < onset)] = w

t0n = np.where(t0 >= 1, t0 + 1, -1)

In [ ]:
ptyp_lab = labels.get('r10ptyp1') or labels.get('r5ptyp1') or {}
print('PTYP labels:', ptyp_lab)

DB_CODES = [1]
DC_CODES = [2]

penct_t1 = at(A('penct'), t1)
types_t1 = np.column_stack([at(A(f'ptyp{k}'), t1) for k in range(1, 5)])
ptyp_t1 = types_t1[:, 0]

def all_in(codes):
    """현 직장의 모든 플랜(1..penct)이 주어진 유형 코드에 속하는가"""
    ok = penct_t1 >= 1
    for k in range(4):
        has_plan_k = penct_t1 >= k + 1
        ok &= ~has_plan_k | np.isin(types_t1[:, k], codes)
    return ok

is_db = (penct_t1 == 1) & np.isin(ptyp_t1, DB_CODES)   # DB: only one plan
is_dc = all_in(DC_CODES)                                # DC: all in DC accounts

# robustness
is_db_multi  = all_in(DB_CODES)                              # multiple DB
is_dc_single = (penct_t1 == 1) & np.isin(ptyp_t1, DC_CODES)  # one DC plan

print('Overlap DB & DC:', (is_db & is_dc).sum())   # must be zero

# DC account balance
dcbal_mat = np.column_stack([at(A(f'dcbal{k}'), t1) for k in range(1, 5)])
dcbal_t1 = np.nansum(dcbal_mat, axis=1)
dcbal_t1[np.isnan(dcbal_mat[:, 0])] = np.nan

PTYP labels: {1: '1.db', 2: '2.dc', 3: '3.db+dc', 'd': '.d=dk', 'j': '.j=web interview, missing', 'm': '.m=oth missing', 'q': '.q=not asked this wv', 'r': '.r=rf', 's': '.s=no pen info', 'u': '.u=unmarried', 'v': '.v=spouse nr', 'z': '.z=no pen based on penct'}
Overlap DB & DC: 0


In [ ]:
ipen = A('ipen')
y0, y1 = at(ipen, t0), at(ipen, t0n)
pos = (y0 > 0) & (y1 > 0)

rows, m = [], np.ones(N, bool)
def step(name, cond):
    global m
    m = m & cond
    rows.append((name, m.sum(), (m & is_db).sum(), (m & is_dc).sum()))

step('0. All respondents', np.ones(N, bool))
step('1. Born 1931-1953 (HRS/WB/EBB)', df['rabyear'].between(1931, 1953).to_numpy())
step('2. Pension onset observed after a non-receipt wave', (onset >= 1) & prior0)
step('3. Receipt in 2 consecutive waves (t0 defined)', t0 >= 1)
step('4. Exactly one pension at t0', at(A('peni_n'), t0) == 1)
step('5. Working wave before t0 (t1 defined)', t1 >= 1)
step('6. Not self-employed at t1', at(A('slfemp'), t1) != 1)
step('7. Pension on t1 job', at(A('jcpen'), t1) == 1)
m_before8 = m.copy()                                   # robustness check
step('8. DB: single DB plan / DC: all plans DC', is_db | is_dc)
step('9. Positive pension income at t0 and t0+1', pos)

att = pd.DataFrame(rows, columns=['Step', 'N', 'of which DB', 'of which DC'])
print(att.to_string(index=False))

final = m.copy()
print('\nDC with t0 >= wave 5 (IRA withdrawal available):', (final & is_dc & (t0 >= 5)).sum())
print('DB with earnings at t1 non-missing:', (final & is_db & ~np.isnan(at(A('iearn'), t1))).sum())
print('DC with balance at t1 non-missing (summed):', (final & is_dc & ~np.isnan(dcbal_t1)).sum())

print('\nRobustness (same steps, alternative plan definitions):')
print('DB, multiple DB plans allowed:', (m_before8 & is_db_multi & pos).sum())
print('DC, single DC plan only     :', (m_before8 & is_dc_single & pos).sum())

                                              Step     N  of which DB  of which DC
                                0. All respondents 45234         2211         1462
                    1. Born 1931-1953 (HRS/WB/EBB) 19562         1981         1151
2. Pension onset observed after a non-receipt wave  6191         1766         1047
    3. Receipt in 2 consecutive waves (t0 defined)  3738         1244          594
                      4. Exactly one pension at t0  3349         1142          533
            5. Working wave before t0 (t1 defined)  3047         1142          533
                        6. Not self-employed at t1  2786         1135          511
                              7. Pension on t1 job  2267         1135          511
          8. DB: single DB plan / DC: all plans DC  1646         1135          511
         9. Positive pension income at t0 and t0+1  1387          955          432

DC with t0 >= wave 5 (IRA withdrawal available): 366
DB with earnings at t1 non-missin

In [ ]:
keep = m_before8 & pos & (is_db | is_dc | is_db_multi | is_dc_single)

out = pd.DataFrame({
    'hhidpn':        df['hhidpn'].to_numpy(),
    'rabyear':       df['rabyear'].to_numpy(),
    'hacohort':      df['hacohort'].to_numpy(),
    't0':            t0,
    't1':            t1,
    'gap_t0_t1':     np.where((t0 >= 1) & (t1 >= 1), t0 - t1, np.nan),
    'penct_t1':      penct_t1,
    'ptyp1_t1':      ptyp_t1,
    'dcbal_t1':      dcbal_t1,
    'ipen_t0':       y0,
    'ipen_t0n':      y1,
    # sample defining
    'is_db':         is_db,
    'is_dc':         is_dc,
    'is_db_multi':   is_db_multi,
    'is_dc_single':  is_dc_single,
    'dc_ira_ok':     t0 >= 5,
})[keep].reset_index(drop=True)

out['type'] = np.select([out['is_db'], out['is_dc']], ['DB', 'DC'], default='')

# main sample distribution
main = out[out['type'] != '']
print('Main sample:', main['type'].value_counts().to_dict())
print('\nClaiming wave (t0) by type:')
print(pd.crosstab(main['t0'], main['type']))
print('\nCohort by type:')
print(pd.crosstab(main['hacohort'], main['type']))
print('\nGap between t1 and t0 (waves):')
print(main.groupby('type')['gap_t0_t1'].describe())

SAVE = '/content/drive/MyDrive/HRS/sample_ids.parquet'
out.to_parquet(SAVE, index=False)
print('\nSaved', len(out), 'rows to', SAVE)

Main sample: {'DB': 955, 'DC': 432}

Claiming wave (t0) by type:
type   DB  DC
t0           
3     131  26
4     131  40
5     138  30
6     117  42
7      94  43
8      61  44
9      72  27
10     40  23
11     75  65
12     45  38
13     26  21
14     17  20
15      8  13

Cohort by type:
type       DB   DC
hacohort          
0           2    0
1           2    0
2           1    0
3         693  268
4         135   79
5         120   81
6           2    4

Gap between t1 and t0 (waves):
      count      mean       std  min  25%  50%  75%   max
type                                                     
DB    955.0  1.635602  1.092758  1.0  1.0  1.0  2.0  11.0
DC    432.0  1.738426  1.264766  1.0  1.0  1.0  2.0   8.0

Saved 1427 rows to /content/drive/MyDrive/HRS/sample_ids.parquet
